In [1]:
!pip install pip==22.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 8.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 23.1.2
    Uninstalling pip-23.1.2:
      Successfully uninstalled pip-23.1.2


In [2]:
!pip install openai==1.21.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.9/309.9 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.7 MB/s eta 0:00:00


In [3]:
! pip install cohere

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.6/166.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 26.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 4.9 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import cohere
co = cohere.Client('LL72FTs5MBRMT2fh9fS1TidqgfPyALIaulUKlU5O') # This is your trial API key

In [10]:
import pandas as pd
data = pd.read_excel("/content/drive/MyDrive/new_911_data/added_title2.xlsx")
data = data[['Title 2']]
data = data[:100]
data

,Title 2
0,حريق
1,غرق
2,تسمم عذائي
3,اختطاف
4,سرقة
...,...
95,مخالفات مرورية متنوعة
96,مشاكل صحية متنوعة
97,مشاكل أمنية متنوعة
98,مضايقة المواطنين


In [11]:
output= pd.DataFrame(index = range(len(data)), columns=["Title", "convs", "prompt"])
#output.to_csv("/content/drive/MyDrive/new_911_data/command_R/CR+_one_question_convs_257.csv")
#output = pd.read_csv("/content/drive/MyDrive/new_911_data/command_R/CR+_dialect_convs_257.csv")
output

,Title,convs,prompt
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
...,...,...,...
95,NaN,NaN,NaN
96,NaN,NaN,NaN
97,NaN,NaN,NaN
98,NaN,NaN,NaN


In [13]:
%%time
import time
import math
#data = data[164:]
for index, row in data.iterrows():
  #if isinstance(output.at[index,'convs'],str):
    #continue
  try:
    current_title = row['Title 2']
    print(f"index: {index} Title: {current_title}")
    prompt_template = f"""
اكتب 3 محادثات مختلفة بلهجة عامية بين مجيب آلي ذكي متخصص في الطوارئ ومتصل لديه حالة طارئة عن ({current_title}) ، تبدأ المحادثة بكلام المتصل ثم ردود المجيب الآلي، يجب أن تكون ردود المجيب الآلي مفيدة ويقوم بطرح سؤال مهم عن الحالة في كل مرة ، يجب أن يقوم بسؤال المتصل عن موقعه الحالي وارسال فرق المساعدة إليه، ويقوم المتصل بذكر عنوان تفصيلي، وبالنهاية يجب أن تحتوي المحادثة على نصائح وارشادات السلامة بشكل تعداد ، لا تضع أي ملاحظات بين أقواس
"""
    print("_______________________________")
    print("Title:", current_title)
    print("_______________________________")
    response = co.generate(
      model='command-r-plus',
      prompt = prompt_template,
      temperature=0.9,
      k=0,
      stop_sequences=[],
      return_likelihoods='NONE'
    )
    print("Response:", response.generations[0].text)
    new_row = pd.DataFrame({'Title': current_title, 'convs':response.generations[0].text}, index=[index])
    output.at[index, 'Title'] = current_title
    output.at[index, 'convs'] = response.generations[0].text
    output.at[index, 'prompt'] = prompt_template
    output.to_csv("/content/drive/MyDrive/new_911_data/command_R/CR+D_100.csv")
    output.to_excel("/content/drive/MyDrive/new_911_data/command_R/CR+D_100.xlsx")
    #save_output.append(new_row)
    time.sleep(60)
  except Exception as e:
    print(e)
    print(index)
    index = index - 1
    continue

output.to_csv("/content/drive/MyDrive/new_911_data/command_R/CR+D_100.csv")
output.to_excel("/content/drive/MyDrive/new_911_data/command_R/CR+D_100.xlsx")


Streaming output truncated to the last 5000 lines.
المتصل: اه، هو واعي بس مش قادر يتكلم كويس، وبيحس باختناق، وكمان جسمه بقى كله كانه مخدر!

المجيب الآلي: طيب، انت فين دلوقتي؟ لو تقدر توصفلي مكانك بالظبط، هبعتلّك اسعاف حالا، ولحد ما يوصل، لازم تساعد اخوك وتخليه مستلقي على ضهره، وتخلي راسه مرفوعة شوية، عشان تسهل عملية التنفس.

المتصل: انا في شارع النصر، في الاسكندرية، في منطقة سيدي جابر، عند محل اسمه "كايرو ماركت".

المجيب الآلي: تمام، وصفك واضح، الاسعاف هيتحرك حالا، ولحد ما يوصل، حاول تهدى اخوك وتطمنه، وكمان ممكن تجيب مروحة او تفتح شبابيك عشان تسهل عملية التنفس، وماتخليهش يشرب اي حاجة.

المحادثة الثالثة:

المتصل: الو، انا عايز مساعدة، انا في حفلة، واحد من اصحابي خد جرعة مخدرات زيادة، ووقع على الارض ومش قادر يقوم!

المجيب الآلي: طيب، حاول تهدى، واخبرني، هل صديقك بيحس بايه دلوقتي؟ هل هو واعي؟

المتصل: والله مش عارف، هو مش قادر يتكلم، وبيحس بدوخة وبيحاول يتقيأ!

المجيب الآلي: طيب، اوصفلي مكانك بالتفصيل، عشان ابعتلك فريق طبي حالا، وكمان عشان سلامة صديقك، ياريت تخليه مستلقي على جنبه، عشان لو

In [9]:
output.isnull().sum()

Title     258
convs     258
prompt    258
dtype: int64

In [10]:
print(response.generations[0].text)

NameError: name 'response' is not defined